# 🗺️ Ejemplo de Geolocalización con Geopy en Jupyter

In [1]:
%pip install geopy pandas folium

Note: you may need to restart the kernel to use updated packages.


## 1. Cargar archivo CSV de direcciones

In [2]:
import pandas as pd

# Leer archivo CSV con direcciones
df = pd.read_csv("direcciones_entrada.csv")
df.head()


,ID,Direccion,Atributo_1,Atributo_2,Atributo_3
0,1,"Av Corrientes 1234, Buenos Aires",13000,0.6,3
1,2,"Florida 500, Buenos Aires",9000,0.6,4
2,3,"Lavalle 1870, Buenos Aires",21000,0.9,5
3,4,"Suipacha 560, Buenos Aires",15000,0.2,2
4,5,"Viamonte 700, Buenos Aires",25000,0.3,4


## 2. Geocodificación con Geopy

In [3]:
from geopy.geocoders import Nominatim
import time

# Crear geolocalizador
geolocator = Nominatim(user_agent="geo_notebook")

# Función para obtener latitud y longitud
def geocode(address):
    try:
        location = geolocator.geocode(address)
        if location:
            return pd.Series([location.latitude, location.longitude])
    except:
        return pd.Series([None, None])

# Aplicar geocodificación a cada fila
df[["Latitud", "Longitud"]] = df["Direccion"].apply(geocode)

# Mostrar resultados
df.head()


,ID,Direccion,Atributo_1,Atributo_2,Atributo_3,Latitud,Longitud
0,1,"Av Corrientes 1234, Buenos Aires",13000,0.6,3,-34.604462,-58.395836
1,2,"Florida 500, Buenos Aires",9000,0.6,4,-34.602397,-58.375328
2,3,"Lavalle 1870, Buenos Aires",21000,0.9,5,-34.603259,-58.393511
3,4,"Suipacha 560, Buenos Aires",15000,0.2,2,-34.601787,-58.379632
4,5,"Viamonte 700, Buenos Aires",25000,0.3,4,-34.599987,-58.376699


## 3. Guardar archivo con coordenadas

In [4]:
df.to_csv("direcciones_geocodificadas_nominatim.csv", index=False)
print("Archivo guardado como 'direcciones_geocodificadas.csv'")


Archivo guardado como 'direcciones_geocodificadas.csv'


## 4. Visualización de los puntos en un mapa interactivo con Folium

In [5]:
import folium

# Crear el mapa centrado en Buenos Aires
mapa = folium.Map(location=[-34.60, -58.38], zoom_start=12)

# Agregar marcadores
for _, row in df.dropna(subset=["Latitud", "Longitud"]).iterrows():
    folium.Marker(
        location=[row["Latitud"], row["Longitud"]],
        popup=row["Direccion"]
    ).add_to(mapa)

# Mostrar mapa
mapa


## 5. Visualización de un Heatmap

In [6]:
import folium

from folium.plugins import HeatMap

# Filtrar filas con coordenadas válidas y atributo 3 no nulo
heat_data = df.dropna(subset=["Latitud", "Longitud", "Atributo_3"])[["Latitud", "Longitud", "Atributo_3"]].values.tolist()

# Crear mapa centrado en Buenos Aires
heatmap_mapa = folium.Map(location=[-34.60, -58.38], zoom_start=12)

# Agregar HeatMap
HeatMap(heat_data, radius=15).add_to(heatmap_mapa)

# Mostrar mapa
heatmap_mapa

## 6. Mapa de Marcadores con valores

In [7]:
import folium

# Crear el mapa centrado en Buenos Aires
mapa_tamanos = folium.Map(location=[-34.60, -58.38], zoom_start=12)

# Agregar marcadores con tamaño basado en Atributo_2
for _, row in df.dropna(subset=["Latitud", "Longitud", "Atributo_2"]).iterrows():
    folium.CircleMarker(
        location=[row["Latitud"], row["Longitud"]],
        radius=row["Atributo_2"] * 50,  # Ajusta el rango de tamaño si lo deseas
        popup=f'{row["Direccion"]}<br>Atributo_1: {row["Atributo_1"]}',
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.6
    ).add_to(mapa_tamanos)

# Mostrar mapa
mapa_tamanos